In [ ]:
import os
import nd2reader
import numpy as np
import pandas as pd
from typing import List, Tuple, Optional, Union, Dict
try:
    from cellpose import models
except Exception as e:
    raise RuntimeError("Cellpose is required: pip install cellpose[gui]") from e
import matplotlib.pyplot as plt
from skimage import measure
from skimage import transform
from skimage.measure import regionprops
from tqdm.notebook import tqdm
plt.rcParams['figure.dpi'] = 150
%matplotlib inline

In [ ]:
filename = "/path/to/experiment_file.nd2"
experiment_name = "local_folder_for_results"
nd2_data = nd2reader.ND2Reader(filename)

N_frames = nd2_data.sizes['t']
N_series = nd2_data.sizes['v']

print(f"Experiment: {experiment_name}")
print(f"Frames: {N_frames}, Series: {N_series}")

In [ ]:
# Create a folder for Cellpose segmentation output
if not os.path.exists(f'cellpose_masks/{experiment_name}'):
    os.makedirs(f'cellpose_masks/{experiment_name}')
# Create folders for json files (segmentation masks) and tables (cell measurements/outputs)
if not os.path.exists(f'js/{experiment_name}/'):
    os.makedirs(f'js/{experiment_name}/', exist_ok=True)
if not os.path.exists(f'tables/{experiment_name}/'):
    os.makedirs(f'tables/{experiment_name}/', exist_ok=True)

# Image functions

In [ ]:
def rescale_image(img, low=2, high=98):
    """Rescale intensities between the `low` and `high` percentiles to 0-1."""
    if low <= 0 or high >= 100:
        return img
    vmin, vmax = np.percentile(img, (low, high))
    img_rescaled = np.clip((img - vmin) / (vmax - vmin + 1e-8), 0, 1)
    return img_rescaled

def gamma_correct(img, gamma=0.8):
    """Apply gamma correction to an image normalized to 0-1."""
    img = np.clip(img, 0, 1)
    if gamma == 1.0:
        return img
    return np.power(img, gamma)

def enhance_contrast(img, low=2, high=98, gamma=0.8):
    """Enhance contrast by rescaling and gamma correction."""
    img_rescaled = rescale_image(img, low, high)
    img_gamma = gamma_correct(img_rescaled, gamma)
    return img_gamma

def get_image(v, t, c, nd2_data=nd2_data):
    return nd2_data.get_frame_2D(t=t, v=v, c=c)

def get_phase(v, t, nd2_data=nd2_data):
    return get_image(v, t, 0, nd2_data)

def get_gfp(v, t, nd2_data=nd2_data):
    return get_image(v, t, 1, nd2_data)

def get_rfp(v, t, nd2_data=nd2_data):
    return get_image(v, t, 2, nd2_data)

def get_merged_fl(v, t, nd2_data=nd2_data):
    rfp = get_rfp(v, t, nd2_data).astype(np.float32)
    gfp /= gfp.max()
    rfp /= rfp.max()
    merged_fl = gfp + rfp
    merged_fl /= merged_fl.max()
    return merged_fl


In [ ]:
from matplotlib.colors import LinearSegmentedColormap

# gfp colormap
b_t_g = [[0, i, 0, 1] for i in np.linspace(0, 1, 256)]
black_to_green = LinearSegmentedColormap.from_list("black_to_green", b_t_g)
# rfp colormap
b_t_r = [[i, 0, 0, 1] for i in np.linspace(0, 1, 256)]
black_to_red = LinearSegmentedColormap.from_list("black_to_red", b_t_r)

# Segment on fluorescence images

In [ ]:
# Initialize cellpose model
model_type = 'bact_fluor_cp3' # name may change based on your Cellpose version
channels = (1, 0)
try:
    _ = models.use_gpu()
    use_gpu = True
except Exception:
    use_gpu = False
try:
    model = models.CellposeModel(gpu=use_gpu, model_type=model_type)
except Exception:
    model = models.Cellpose(gpu=use_gpu, model_type=model_type)

# Skip series with existing masks
# TODO: add option to overwrite
series_tasks = [
    int(v) for v in range(N_series)
    if not os.path.exists(f'cellpose_masks/{experiment_name}/masks_v{v}.npz')
]

nd2_data = nd2reader.ND2Reader(filename)
with tqdm(total=N_series,
          initial=min(series_tasks),
          desc='Segmenting cells',
          dynamic_ncols=True,
          unit='series') as pbar:
    pbar.refresh()
    for v in series_tasks:        
        # Load merged gfp/rfp images for this series
        images = [get_merged_fl(v, t, nd2_data) for t in range(N_frames)]
        
        # Segment images
        masks, *_ = model.eval(
            images,
            channels=channels,
            do_3D=False,
            batch_size=16, # optimize based on available GPU memory/CPU threads
            
            # optimize segmentation parameters as needed (defaults are usually good for bacteria)
            flow_threshold=0.5,
            cellprob_threshold=-1.0,
        )
        
        # Stack masks into a single array for compressed storage
        masks_stacked = np.vstack([m[np.newaxis, ...] for m in masks]) # shape (N_frames, H, W)
        
        # Save masks
        filepath = f'cellpose_masks/{experiment_name}/masks_v{v}.npz'
        np.savez_compressed(filepath, masks_stacked)
        
        pbar.update(1)

# Define all series lane ROIs

In [ ]:
import os, json

# Load existing crops if available, otherwise start with empty dict
if os.path.exists(f'js/{experiment_name}/crops.json'):
    with open(f'js/{experiment_name}/crops.json', 'r') as f:
        series_to_crop = json.load(f)
        series_to_crop = {int(k): v for k, v in series_to_crop.items()}
else:
    series_to_crop = {}

print(f"{len(series_to_crop)} crops defined.")

## Create a widget to quickly create mother machine lane masks (rectangle bands)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

def compute_roi(v, crop1, crop2, crop3, crop4, crop5, angle, H, W):
    roi = np.zeros((H, W), dtype=float)
    x = crop3
    while x + crop4 < W:
        roi[int(crop1):int(crop1+crop2), int(x):int(x+crop4)] = 1
        x += crop5
    roi = transform.rotate(roi, angle).astype(bool)
    return roi

def get_lane_normal(lane_id, crop1, crop2, crop3, crop4, crop5, angle, flip_Y, H, W):
    """
    Get the normal vector of the lane line.
    """

    # get the position vector of the lane initial point
    # from the center of the image
    p0 = np.array([
        (crop3 + lane_id * crop5 + 0.5 * crop4) - W / 2 + 0.5,
        (crop1 + crop2) - H / 2 + 0.5
    ]) if not flip_Y else np.array([
        (crop3 + lane_id * crop5 + 0.5 * crop4) - W / 2 + 0.5,
        (crop1) - H / 2 + 0.5
    ])

    # rotate the position vector by the given angle
    angle_rad = np.radians(angle)
    c = np.cos(angle_rad)
    s = np.sin(angle_rad)
    M = np.array([
        [c, s],
        [-s, c]
    ])
    p1 = M @ p0
    
    # compute the unit normal vector along the lane
    n = np.array([-s, -c]) if not flip_Y else np.array([s, c])
    n = n / np.linalg.norm(n)
    
    p2 = p1 + np.array([W/2 - 0.5, H/2 - 0.5])

    return p2, n

def test_series_crop(v, crop):
    image = nd2_data.get_frame_2D(t=0, v=v, c=0).astype(np.float32)
    roi = compute_roi(v, *crop[:-1], *image.shape)

    _, axs = plt.subplots(1, 2, figsize=(6, 3))

    eim = enhance_contrast(image, 2, 98, 0.8)
    axs[0].imshow(eim, cmap='gray')

    eim[~roi] = 0
    axs[1].imshow(eim, cmap='gray')
    axs[1].set_axis_off()

    p0, n = get_lane_normal(5, *crop, *image.shape)

    axs[1].quiver(
        p0[0], p0[1],
        n[0]*crop[1], n[1]*crop[1],
        color='red', angles='xy', scale_units='xy', scale=1, width=0.005
    )

    series_to_crop[v] = crop

    plt.tight_layout()
    plt.show()


# Dropdown for selecting series v
v_dropdown = widgets.Dropdown(
    options=list(range(N_series)),
    value=0,
    description='Series v',
)

# Number inputs for crop parameters (max values updated dynamically per series)
crop1 = widgets.BoundedFloatText(description='row', min=0, max=1000, step=0.01, value=700)
crop2 = widgets.BoundedFloatText(description='height', min=1, max=1000, step=0.01, value=300)
crop3 = widgets.BoundedFloatText(description='x0', min=0, max=1000, step=0.01, value=10)
crop4 = widgets.BoundedFloatText(description='width', min=1, max=1000, step=0.01, value=25)
crop5 = widgets.BoundedFloatText(description='stride', min=1, max=1000, step=0.01, value=65)
angle = widgets.BoundedFloatText(description='angle', min=-180, max=180, step=0.01, value=0)
flip_Y = widgets.Checkbox(description='flip Y', value=False)

# Single output area for the plot
out = widgets.Output()

def _draw(*args):
    with out:
        clear_output(wait=True)
        test_series_crop(
            v_dropdown.value,
            [crop1.value,
            crop2.value,
            crop3.value,
            crop4.value,
            crop5.value,
            angle.value,
            flip_Y.value]
        )

# Update ranges and default values when v changes
def _update_ranges_and_defaults(change=None):
    v = v_dropdown.value
    image = nd2_data.get_frame_2D(t=0, v=v, c=0)
    H, W = image.shape

    crop1.max = max(0, H - 1)
    crop2.max = max(1, H)
    crop3.max = max(0, W - 1)
    crop4.max = max(1, W)
    crop5.max = max(1, W)
    
    if v in series_to_crop:
        c1, c2, c3, c4, c5, a, flip = series_to_crop[v]
    else:
        c1 = min(crop1.value, max(0, H - 1))
        c2 = min(crop2.value, max(1, H))
        c3 = min(crop3.value, max(1, W))
        c4 = min(crop4.value, max(1, W))
        c5 = min(crop5.value, max(1, W))
        a = angle.value
        flip = flip_Y.value
        
    # Clamp to valid ranges before assigning; suppress redraws during set
    v1 = np.clip(c1, crop1.min, crop1.max)
    v2 = np.clip(c2, crop2.min, crop2.max)
    v3 = np.clip(c3, crop3.min, crop3.max)
    v4 = np.clip(c4, crop4.min, crop4.max)
    v5 = np.clip(c5, crop5.min, crop5.max)
    v6 = np.clip(a, angle.min, angle.max)

    if crop1.value != v1:
        crop1.value = v1
    if crop2.value != v2:
        crop2.value = v2
    if crop3.value != v3:
        crop3.value = v3
    if crop4.value != v4:
        crop4.value = v4
    if crop5.value != v5:
        crop5.value = v5
    if abs(angle.value - v6) > 0.01:
        angle.value = v6
    if flip_Y.value != flip:
        flip_Y.value = flip

    _draw()


# Observe changes: only crops trigger redraw; dropdown triggers range update
for w in (crop1, crop2, crop3, crop4, crop5, angle, flip_Y):
    w.observe(_draw, names='value')

v_dropdown.observe(_update_ranges_and_defaults, names='value')

# Initialize ranges/values and render once
_update_ranges_and_defaults()

# Layout controls and live output
controls = widgets.VBox([v_dropdown, crop1, crop2, crop3, crop4, crop5, angle, flip_Y])
ui = widgets.HBox([controls, out])
display(ui)

In [ ]:
# Run this cell after adjusting crops for all series to save the crops and compute ROIs
with open(f'js/{experiment_name}/crops.json', 'w') as f:
    json.dump(series_to_crop, f)

series_to_roi = {
    i: compute_roi(
        i, 
        *series_to_crop[i][:-1], 
        *nd2_data.get_frame_2D(t=0, v=i, c=0).shape
)
    for i in series_to_crop
}

# Compute statistics from segmentation masks

In [ ]:
def compute_mean_sem(values: np.ndarray,
                     min_n: int = 10) -> Tuple[float, float]:
    """Detect and handle saturation, then compute mean and SEM using censored-normal MLE if needed."""
    if values.size < min_n:
        return -1.0, -1.0

    vals_raw = np.asarray(values)

    # Detect saturation at dtype max for integer arrays
    if np.issubdtype(vals_raw.dtype, np.integer):
        max_val = np.iinfo(vals_raw.dtype).max
        num_censored = int(np.count_nonzero(vals_raw == max_val))
    else:
        max_val = None
        num_censored = 0

    # If no saturated pixels, return simple mean and SEM over all values
    if num_censored == 0:
        vals = np.sort(vals_raw.astype(float))
        mean = float(np.mean(vals))
        sem = float(np.std(vals, ddof=1) / np.sqrt(vals.size))
        return mean, sem

    # Otherwise, mask saturated pixels and compute censored-normal MLE
    mask = vals_raw != max_val
    vals = vals_raw[mask].astype(float)

    if vals.size == 0:
        return -1.0, -1.0
    if vals.size == 1:
        mean = float(vals[0])
        sem = 0.0
        return mean, sem

    t = float(max_val)
    sqrt2 = np.sqrt(2.0)

    def loglike(mu: float, sigma: float) -> float:
        if not np.isfinite(mu) or not np.isfinite(sigma) or sigma <= 1e-12:
            return -np.inf
        # Survival function P(X > t) using erfc for numerical stability
        z = (t - mu) / sigma
        try:
            from math import erfc as _erfc
        except Exception:
            from scipy.special import erfc as _erfc
        sf = 0.5 * _erfc(z / sqrt2)
        sf = np.clip(sf, 1e-300, 1.0)
        # Uncensored log-pdf terms
        ll_unc = -np.log(sigma) - 0.5 * np.log(2.0 * np.pi) - 0.5 * ((vals - mu) / sigma) ** 2
        return float(np.sum(ll_unc) + num_censored * np.log(sf))

    mu_center = float(np.mean(vals))
    sigma_center = float(np.std(vals, ddof=1))
    if not np.isfinite(sigma_center) or sigma_center <= 1e-6:
        sigma_center = max(float(np.std(vals)), 1e-3)

    # Coordinate search to maximize censored-normal log-likelihood
    for k in range(4):
        mu_span = 3.0 * sigma_center / (k + 1)
        mu_candidates = np.linspace(mu_center - mu_span, mu_center + mu_span, 25)
        best_ll = -np.inf
        best_mu = mu_center
        for mu_c in mu_candidates:
            ll = loglike(mu_c, sigma_center)
            if ll > best_ll:
                best_ll = ll
                best_mu = mu_c
        mu_center = best_mu

        sigma_factors = np.exp(np.linspace(-1.0, 1.0, 21) / (k + 1))
        best_ll = -np.inf
        best_sigma = sigma_center
        for sfac in sigma_factors:
            s_c = max(sigma_center * sfac, 1e-6)
            ll = loglike(mu_center, s_c)
            if ll > best_ll:
                best_ll = ll
                best_sigma = s_c
        sigma_center = best_sigma

    mu_hat = float(mu_center)
    sigma_hat = float(sigma_center)
    sem = float(sigma_hat / np.sqrt(vals.size))
    return mu_hat, sem


def build_lane_map(image_shape,
                   crop1, crop2, crop3, crop4, crop5,
                   angle) -> np.ndarray:
    """
    Create an integer lane_map array of shape (H, W) where each pixel is:
    -1 if not in any rotated ROI,
     i if pixel belongs to lane i (first non-overlapping ROI wins)

    image_shape: (H, W)
    crop1, crop2: top and height of the unrotated box
    crop3, crop4: initial left and width of the unrotated box
    crop5: stride in x
    angle: rotation angle in degrees
    """
    H, W = image_shape
    lane_map = np.full((H, W), fill_value=-1, dtype=np.int16)

    x = crop3
    lane_idx = 0
    while x + crop4 <= W:
        roi = np.zeros((H, W), dtype=np.uint8)
        roi[int(crop1):int(crop1 + crop2), int(x):int(x + crop4)] = 1

        roi_rot = transform.rotate(
            roi,
            angle=angle,
            order=0,
            preserve_range=True,
            resize=False
        ).astype(bool)

        update_mask = roi_rot & (lane_map < 0)
        lane_map[update_mask] = lane_idx

        lane_idx += 1
        x += crop5

    return lane_map

In [ ]:
# Assuming 0.5 um per pixel at 20X, adjust for actual magnification if needed (e.g. 150X here)
pixel_length_um_20X = 0.5
pixel_length_um = pixel_length_um_20X * 20 / 150
pixel_area_um2 = pixel_length_um**2

# Iterate over all series and frames to compute statistics for each segmented cell
statistics_rows = []
for v in tqdm(series_to_crop,
              desc='Computing statistics',
              dynamic_ncols=True,
              unit='series'):
    masks_path = f'cellpose_masks/{experiment_name}/masks_v{v}.npz'
    masks = np.load(masks_path)['arr_0']  # shape (N_frames, H, W)

    # build rotated ROIs for lane classification
    crop = series_to_crop[v]
    lane_map = build_lane_map(
        masks[0].shape,
        *crop[:-1]
    )
    n_lanes = np.max(lane_map) + 1
    lane_normals = [
        get_lane_normal(i, *crop, *lane_map.shape) 
        for i in range(n_lanes)
    ]

    # iterate timepoints
    total_frames = masks.shape[0]
    for t, mask in tqdm(enumerate(masks),
                        desc=f'Series {v}',
                        dynamic_ncols=True,
                        position=1,
                        total=total_frames,
                        unit='frame'):
        _nd2 = nd2_data
        _t = t
        gfp = get_gfp(v, _t, nd2_data=_nd2)
        rfp = get_rfp(v, _t, nd2_data=_nd2)
        gfp_bg = np.median(gfp[-100:, -100:])
        rfp_bg = np.median(rfp[-100:, -100:])
        gfp -= gfp_bg.astype(gfp.dtype)
        rfp -= rfp_bg.astype(rfp.dtype)


        # regionprops once per frame for geometry (centroid, major axis length)
        props = regionprops(mask)

        for p in props:
            cell_id = p.label
            coords = p.coords  # (Npix, 2) (row, col)

            rr = coords[:, 0]
            cc = coords[:, 1]

            # intensities for this cell
            cell_gfp = gfp[rr, cc]
            cell_rfp = rfp[rr, cc]

            gfp_mean, gfp_mean_err = compute_mean_sem(cell_gfp)
            rfp_mean, rfp_mean_err = compute_mean_sem(cell_rfp)

            # centroid and "length"
            cy, cx = p.centroid if hasattr(p, "centroid") else (np.nan, np.nan)
            length = p.major_axis_length if hasattr(p, "major_axis_length") else np.nan
            width = p.minor_axis_length if hasattr(p, "minor_axis_length") else np.nan
            area = p.area if hasattr(p, "area") else np.nan
            eccentricity = p.eccentricity if hasattr(p, "eccentricity") else np.nan
            orientation = p.orientation if hasattr(p, "orientation") else np.nan

            # lane via majority vote of lane_map pixels
            lanes_for_pixels = lane_map[rr, cc]
            valid = lanes_for_pixels[lanes_for_pixels >= 0]

            if valid.size == 0 or n_lanes == 0:
                lane_idx = -1
            else:
                counts = np.bincount(valid, minlength=n_lanes)
                lane_idx = int(np.argmax(counts))
            
            if lane_idx == -1:
                lane_pos = -1
            else:
                p0, n = lane_normals[lane_idx]
                lane_pos = np.dot(np.array([cx, cy]) - p0, n) / crop[1]
            
            if lane_pos > 0.9:
                continue
            lane_pos /= 0.9

            statistics_rows.append({
                'v': int(v),
                't': int(t),
                'id': int(cell_id),
                'gfp_mean': float(gfp_mean),
                'gfp_mean_err': float(gfp_mean_err),
                'rfp_mean': float(rfp_mean),
                'rfp_mean_err': float(rfp_mean_err),
                'gfp_rfp_ratio': float(gfp_mean / rfp_mean),
                'gfp_rfp_ratio_err': float(np.hypot(gfp_mean_err, rfp_mean_err)),
                'lane': int(lane_idx),
                'centroid_u': float(lane_pos),
                'centroid_x': float(cx) * pixel_length_um,
                'centroid_y': float(cy) * pixel_length_um,
                'cell_length': float(length) * pixel_length_um,
                'cell_width': float(width) * pixel_length_um,
                'cell_area': float(area) * pixel_area_um2,
                'cell_eccentricity': float(eccentricity),
                'orientation': float(orientation)
            })

# Create DataFrame and save to CSV
df = pd.DataFrame(
    statistics_rows,
    columns=[
        'v', 't', 'id',
        'gfp_mean', 'gfp_mean_err',
        'rfp_mean', 'rfp_mean_err',
        'gfp_rfp_ratio', 'gfp_rfp_ratio_err',
        'lane', 'centroid_u',
        'centroid_x', 'centroid_y',
        'cell_length', 'cell_width',
        'cell_area', 'cell_eccentricity',
        'orientation'
    ]
)

output_csv = f'tables/{experiment_name}/statistics.csv'
df.to_csv(output_csv, index=False)